# Практическая работа №3. Сборка пайплайна предобработки

## Цель работы

Собрать пайплайн предобработки данных, исключающий утечку информации из тестовой выборки в обучающую.

## Вариант

Уровень B.

## Датасет

Abalone.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Загрузка исходного датасета

На этом этапе загружается исходный датасет Abalone. Статистические операции для заполнения пропусков пока не выполняются.

In [2]:
df = pd.read_csv("../data/abalone.data.csv")

df.head()

,gender,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## 2. Первичный просмотр данных

Проверим размер датасета, названия столбцов и наличие пропущенных значений.

In [3]:
print("Размер датасета:", df.shape)

print("\nНазвания столбцов:")
print(df.columns.tolist())

print("\nКоличество пропусков:")
print(df.isnull().sum())

Размер датасета: (4177, 9)

Названия столбцов:
['gender', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight', 'Rings']

Количество пропусков:
gender            0
Length            0
Diameter          0
Height            0
Whole weight      0
Shucked weight    0
Viscera weight    0
Shell weight      0
Rings             0
dtype: int64


## 3. Разделение признаков и целевой переменной

Целевой переменной будет `Rings`. Остальные столбцы используются как признаки для дальнейшей предобработки.

In [4]:
X = df.drop(columns=["Rings"])
y = df["Rings"]

print("Размер X:", X.shape)
print("Размер y:", y.shape)

print("\nПризнаки X:")
print(X.columns.tolist())

Размер X: (4177, 8)
Размер y: (4177,)

Признаки X:
['gender', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight']


## 4. Классификация признаков

Категориальным признаком является `gender`, так как он содержит категории M, F и I.

Числовыми признаками являются `Length`, `Diameter`, `Height`, `Whole weight`, `Shucked weight`, `Viscera weight` и `Shell weight`.

Для числовых признаков будем использовать заполнение пропусков медианой и последующее масштабирование. Для категориального признака `gender` будем использовать отдельную категорию для пропущенных значений и One-Hot Encoding.

In [5]:
numeric_features = [
    "Length",
    "Diameter",
    "Height",
    "Whole weight",
    "Shucked weight",
    "Viscera weight",
    "Shell weight"
]

categorical_features = [
    "gender"
]

print("Числовые признаки:")
print(numeric_features)

print("\nКатегориальные признаки:")
print(categorical_features)

Числовые признаки:
['Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight', 'Viscera weight', 'Shell weight']

Категориальные признаки:
['gender']


## 5. Выбор стратегии обработки пропусков

Для числовых признаков используется медиана. Медиана является устойчивой к выбросам и поэтому подходит для биометрических и весовых характеристик датасета Abalone.

Для категориального признака `gender` используется отдельная категория `missing`. Это позволяет не удалять строки с пропущенным значением и не приписывать им искусственную категорию M, F или I.

Среднее значение для числовых признаков не используется, так как оно сильнее зависит от выбросов.

## 6. Разбиение данных на обучающую и тестовую выборки

Разбиение выполняется до обучения объектов предобработки. Это необходимо для предотвращения утечки информации из тестовой выборки.

Тестовая выборка не должна использоваться для вычисления медиан, средних значений или других статистик.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)
print("Размер y_train:", y_train.shape)
print("Размер y_test:", y_test.shape)

Размер X_train: (3341, 8)
Размер X_test: (836, 8)
Размер y_train: (3341,)
Размер y_test: (836,)


## 7. Пайплайн обработки числовых признаков

Для числовых признаков выполняются два шага:

1. Пропуски заменяются медианой, рассчитанной только на обучающей выборке.
2. Значения масштабируются с помощью StandardScaler.

Оба действия выполняются внутри Pipeline.

In [8]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

## 8. Пайплайн обработки категориального признака

Для `gender` пропущенные значения заменяются отдельной категорией `missing`.

После этого категориальный признак преобразуется в числовой формат с помощью One-Hot Encoding.

In [9]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

## 9. Объединение преобразований с помощью ColumnTransformer

ColumnTransformer позволяет применить разные наборы преобразований к разным группам признаков.

Числовые признаки проходят через `numeric_pipeline`, а категориальный признак `gender` проходит через `categorical_pipeline`.

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

## 10. Итоговый пайплайн предобработки

Все операции предобработки объединяются в единый Pipeline. Это позволяет обучить преобразования только на обучающей выборке и затем применить те же преобразования к тестовой выборке без повторного обучения.

In [12]:
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

## 11. Обучение пайплайна только на обучающей выборке

Пайплайн обучается только на `X_train`. Поэтому медианы и параметры масштабирования вычисляются исключительно по обучающей выборке.

Тестовая выборка не участвует в процессе обучения.

In [13]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

In [14]:
X_test_processed = preprocessing_pipeline.transform(X_test)

## 12. Проверка результатов предобработки

Сравним размеры исходных обучающей и тестовой выборок с размерами данных после преобразования.

In [15]:
print("До предобработки:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nПосле предобработки:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)

До предобработки:
X_train: (3341, 8)
X_test : (836, 8)

После предобработки:
X_train_processed: (3341, 10)
X_test_processed : (836, 10)


In [16]:
print("Количество строк X_train до:", X_train.shape[0])
print("Количество строк X_train после:", X_train_processed.shape[0])

print("Количество строк X_test до:", X_test.shape[0])
print("Количество строк X_test после:", X_test_processed.shape[0])

Количество строк X_train до: 3341
Количество строк X_train после: 3341
Количество строк X_test до: 836
Количество строк X_test после: 836


In [17]:
feature_names = preprocessing_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

print("Количество признаков после обработки:", len(feature_names))

print("\nНовые признаки:")
print(feature_names)

Количество признаков после обработки: 10

Новые признаки:
['numeric__Length' 'numeric__Diameter' 'numeric__Height'
 'numeric__Whole weight' 'numeric__Shucked weight'
 'numeric__Viscera weight' 'numeric__Shell weight' 'categorical__gender_F'
 'categorical__gender_I' 'categorical__gender_M']


## 13. Проверка отсутствия утечки данных

Утечка данных отсутствует, поскольку:

- сначала данные были разделены на `train` и `test`;
- `Pipeline` был обучен только на `X_train`;
- медиана для числовых признаков рассчитывалась только на обучающей выборке;
- параметры масштабирования рассчитывались только на обучающей выборке;
- `X_test` был только преобразован методом `transform()`;
- тестовая выборка не использовалась для обучения предобработки.

In [19]:
print("Pipeline:")
print(preprocessing_pipeline)

Pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Length', 'Diameter',
                                                   'Height', 'Whole weight',
                                                   'Shucked weight',
                                                   'Viscera weight',
                                                   'Shell weight']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                   

## 14. Вывод

В ходе практической работы был собран единый пайплайн предобработки данных датасета Abalone.

Числовые признаки обрабатываются с помощью заполнения пропусков медианой и стандартизации. Для категориального признака `gender` пропуски заменяются отдельной категорией `missing`, после чего выполняется One-Hot Encoding.

Разбиение на обучающую и тестовую выборки было выполнено до обучения преобразований. Пайплайн был обучен только на `X_train`, после чего к `X_test` применён метод `transform()`.

Таким образом, статистики предобработки не вычислялись на тестовой выборке, поэтому утечка данных из test в train исключена.